#COMPSCI 546: Applied Information Retrieval - Spring 2026 ([website](https://groups.cs.umass.edu/zamani/compsci-546-applied-information-retrieval-spring-2026/))
##Assignment 4: Learning to Rank (Total : 100 points)

**Description**

This assignment consists of programming and analytical questions on Learning to Rank models.

**Instructions**

* To start working on the assignment, you would first need to save the notebook to your local Google Drive. For this purpose, you can click on *Copy to Drive* button. You can alternatively click the *Share* button located at the top right corner and click on *Copy Link* under *Get Link* to get a link and copy this notebook to your Google Drive.  

*   For questions with descriptive answers, please replace the text in the cell which states "Enter your answer here!" with your answer. If you are using mathematical notation in your answers, please define the variables.
*   You should implement all the functions yourself and should not use a library or tool for the computation.
*   For coding questions, you can add code where it says "enter code here" and execute the cell to print the output.
* To create the final pdf submission file, execute *Runtime->RunAll* from the menu to re-execute all the cells and then generate a PDF using *File->Print->Save as PDF*. Make sure that the generated PDF contains all the codes and printed outputs before submission.


**Submission Details**

* Due data: Wednesday, April 8, 2026 at 11:59 PM (EDT).
* The final PDF file must be submitted to Gradescope.
* After copying this notebook to your Google Drive, please paste a link to it below. Use the same process given above to generate a link. ***You will not recieve any credit if you don't paste the link!*** Make sure we can access the file.
***LINK: https://colab.research.google.com/drive/1QNMcXr80hEd8vFGdOd89xTmE7BTjkZB8?usp=sharing***

**Academic Honesty**

Please follow the guidelines under the *Collaboration and Help* section of the course website.     

# Download input files

Please execute the cell below to download the input files.

In [10]:
import os
import zipfile

# 1. Download files
!gdown 11K8r5a_Aj9S4Cpd8k3x76ccBTluNguip -O HW07.zip

# 2. Extract the file (Same as your previous code)
with zipfile.ZipFile('HW07.zip', 'r') as zip_file:
    zip_file.extractall('./')

# 3. Cleanup and Setup
if os.path.exists('HW07.zip'):
    os.remove('HW07.zip')

# We will use hw1 as our working directory
os.chdir('HW07')

Downloading...
From (original): https://drive.google.com/uc?id=11K8r5a_Aj9S4Cpd8k3x76ccBTluNguip
From (redirected): https://drive.google.com/uc?id=11K8r5a_Aj9S4Cpd8k3x76ccBTluNguip&confirm=t&uuid=e65a3408-ad6d-48c3-9c3d-b52567891eb9
To: /content/HW07/HW07/HW07.zip
100% 33.3M/33.3M [00:00<00:00, 71.3MB/s]


In [11]:
#Setting the input files
passage_file = "antique-collection.tok.clean_kstem"
test_queries_file = "antique-test-queries.tok.clean_kstem"
train_queries_file = "antique-train-queries.tok.clean_kstem"
val_queries_file = "antique-val-queries.tok.clean_kstem"
sample = "sample.txt"
stopwords_file = "stopword_INQUERY"
val_baseline_features_file = "val_baseline_features_top10"
test_baseline_features_file = "test_baseline_features_top10"
train_baseline_features_file = "train_baseline_features_top10"

# 1 : Initial Data Setup (30 points)

We use collection from the ANTIQUE  [https://arxiv.org/pdf/1905.08957.pdf] dataset for this assignment. As described in the previous assignments, this is a passage retrieval dataset.

The description of the input files provided for this assignment is given below.

**Collection file**

Each row of the file consists of the following information:

*passage_id  passage_text*

The id and text information is tab separated. The passage text has been pre-processed to remove punctutation, tokenised and stemmed using the Krovetz stemmer. The terms in the passage text can be accessed by splitting the text based on space.

**Query files**

You are provided with train,validation and test query files.  Each row of the file consists of the following information:

*query_id  query_text*

The id and text information is tab separated. The query text has been pre-processed to remove punctutation, tokenised and stemmed using the Krovetz stemmer. The terms in the text can be accessed by splitting the text based on space.

**Feature files**

You are provided with train,validation and test feature files. Each row of the file consists of the following information:

*query_id  passage_id relevance_score vsm_score bm25_score*

Each row contains features for a (query,passage) pair and is space separated. The relevance_score is the human annotated relevance score. vsm_score and bm25 scores are the relevance scores for the pair corresponding to the two different scoring methods.

**Stopwords file**

The stopword file contains the list of stopwords. This file has a stopword per line.

**Sample file**

For this assignment, we use the pyltr Learning to Rank framework from [https://github.com/harshhpareek/pyltr]. The input file to the framework has to be set similar to the sample.txt in the following format:

*relevance_score qid:query_id 1:feature1 2:feature2 #docid = passage_id*

Each entry is space separated. This file has been provided only for reference to create files of the same format.

In the cell below, you have to implement the following:


*   Load the query files
*   Load the collection
*   Load the stopwords




In [12]:

'''
In this function, load the query files into dict
Return Variables:
queries - dict with qid as key and querytext as value
'''
def loadQueryFile(filename):
    #enter your code here
    queries = {}
    with open(filename, 'r', encoding='utf-8') as file:
      for line in file:
        line = line.strip()
        if not line:
            continue

        qid, query_text = line.strip().split('\t')
        queries[qid] = query_text

    return queries


'''
In this function, load the collection into dict
Return Variables:
coll - dict with passage id as key and passage text as value
'''
def loadCollection(passage_file):
    #enter your code here
    coll = {}
    with open(passage_file, 'r', encoding='utf-8') as file:
      for line in file:
        line = line.strip()
        if not line:
            continue

        passage_id, passage_text = line.strip().split('\t')
        coll[passage_id] = passage_text

    return coll

'''
In this function, load the stopwords into dict
Return Variables:
stopwords - dict with stopword as key
'''
def loadStopWords(stopwords_file):
    #enter your code here
    stopwords = {}
    with open(stopwords_file, 'r', encoding='utf-8') as file:
      for line in file:
        line = line.strip()
        if not line:
            continue

        stopword = line.strip()
        stopwords[stopword] = 1

    return stopwords


train_queries = loadQueryFile(train_queries_file)
val_queries = loadQueryFile(val_queries_file)
test_queries = loadQueryFile(test_queries_file)
coll = loadCollection(passage_file)
stopwords = loadStopWords(stopwords_file)

print('Total Number of train queries: {0}'.format(len(train_queries)))
print('Total Number of validation queries: {0}'.format(len(val_queries)))
print('Total Number of test queries: {0}'.format(len(test_queries)))
print('Total Number of passages in the collection: {0}'.format(len(coll)))
print('Total Number of stopwords: {0}'.format(len(stopwords)))


Total Number of train queries: 2226
Total Number of validation queries: 200
Total Number of test queries: 200
Total Number of passages in the collection: 403492
Total Number of stopwords: 418


# 2 : Feature Preparation (30 points)

The input feature file consists of two main features : VSM score and bm25 score of query,passage pairs. In this section, you will implement three additional features and use the information to create input feature file which contains the 5 features. The feature file must have the same format as sample.

In the cell below, implement the following features:

*  Number of unique term overlap between query and passage after excluding stopwords and words with only one character from both.

  [ Example = Query : why do a cat headbutt

  Passage : cat fight for attention and domination if you show a can of food to my cat he headbutt it.

  Number of Overlapped terms for the query/passage pair: 2  ]

*  Number of terms in query
*  Number of terms in passage


In [13]:
'''
In this function, create new feature file with additional features in the format required as input
Return Variables:
There is no return variable. You would create a new feature file "final_features_file"
One example of the row of the newly created file is
"0 qid:3990512 1:3.5053628162466897 2:10.841493137122296 3:1 4:6 5:112 #docid = 882429_10"
The format of the file is:
"relevance_score qid:query_id 1:feature1 2:feature2 3:feature3 4:feature4 5:feature5 #docid = passage_id"
You can read through the input baseline_features_file, create additional features and
add the updated entry into the new file.
'''

def featureCreation(baseline_features_file, stopwords, queries, coll, final_features_file):
    # make stopwords usable whether it is a dict or set
    stopword_set = set(stopwords) if not isinstance(stopwords, dict) else set(stopwords.keys())

    with open(baseline_features_file, 'r', encoding='utf-8') as infile:
      with open(final_features_file, 'w', encoding='utf-8') as outfile:
        for line in infile:
          line = line.strip()
          if not line:
              continue

          query_id, passage_id, relevance_score, vsm_score, bm25_score = line.split()

          query_text = queries[query_id]
          passage_text = coll[passage_id]

          query_terms = query_text.split()
          passage_terms = passage_text.split()

          # for overlap feature:
          # exclude stopwords and one-character words, then count unique overlap
          filtered_query_terms = {
              term for term in query_terms
              if term not in stopword_set and len(term) > 1
          }

          filtered_passage_terms = {
              term for term in passage_terms
              if term not in stopword_set and len(term) > 1
          }

          num_overlap_term = len(filtered_query_terms & filtered_passage_terms)
          # number of terms in query/passage = token counts
          num_query_term = len(query_terms)
          num_passage_term = len(passage_terms)
          new_line = (
              f"{relevance_score} "
              f"qid:{query_id} "
              f"1:{vsm_score} "
              f"2:{bm25_score} "
              f"3:{num_overlap_term} "
              f"4:{num_query_term} "
              f"5:{num_passage_term} "
              f"#docid = {passage_id}"
          )
          outfile.write(new_line + "\n")

featureCreation(train_baseline_features_file, stopwords, train_queries, coll, 'train_features_final')
featureCreation(val_baseline_features_file, stopwords, val_queries, coll, 'val_features_final')
featureCreation(test_baseline_features_file, stopwords, test_queries, coll, 'test_features_final')

# 3 : Model Training and Evaluation (30 points)

In the cell below, the pyltr library is used to train and evaluate ANTIQUE data using LambdaMART model. This takes less than a minute to execute.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import lightgbm as lgb

def read_letor_file(filepath):
    X, y, qids = [], [], []
    with open(filepath) as f:
        for line in f:
            parts = line.strip().split()
            y.append(float(parts[0]))
            qid = parts[1].split(':')[1]
            qids.append(qid)
            features = []
            for p in parts[2:]:
                if p.startswith('#'):
                    break
                features.append(float(p.split(':')[1]))
            X.append(features)
    return np.array(X), np.array(y), np.array(qids)

# Load train / validation / test sets
TX, Ty, Tqids = read_letor_file('train_features_final')
VX, Vy, Vqids = read_letor_file('val_features_final')
EX, Ey, Eqids = read_letor_file('test_features_final')

# Grouping rows by query
def get_group_sizes(qids):
    groups = []
    current, count = qids[0], 0
    for q in qids:
        if q == current:
            count += 1
        else:
            groups.append(count)
            current, count = q, 1
    groups.append(count)
    return groups

# Build LightGBM dataset
train_ds = lgb.Dataset(TX, label=Ty, group=get_group_sizes(Tqids))
val_ds = lgb.Dataset(VX, label=Vy, group=get_group_sizes(Vqids), reference=train_ds)

# Training parameters
params = {
    'objective': 'lambdarank', # Train a ranking model, not regression/classification
    'metric': 'ndcg', # use NDCG metric during training
    'ndcg_eval_at': [10],
    'n_estimators': 100,
    'learning_rate': 0.02,
    'num_leaves': 10,
    'min_data_in_leaf': 64,
    'feature_fraction': 0.5,
    'verbose': -1,
}

model = lgb.train(
    params, train_ds,
    num_boost_round=100,
    valid_sets=[val_ds],
    callbacks=[lgb.early_stopping(stopping_rounds=250, verbose=True)]
)

Epred = model.predict(EX)

# Calculate NDCG@10
from sklearn.metrics import ndcg_score
unique_qids = list(dict.fromkeys(Eqids))
ndcg_scores = []
idx = 0
for qid in unique_qids:
    count = np.sum(Eqids == qid)
    true = Ey[idx:idx+count]
    pred = Epred[idx:idx+count]
    if len(true) > 1:
        ndcg_scores.append(ndcg_score([true], [pred], k=10))
    idx += count

print('LambdaMART model test score: ' + str(np.mean(ndcg_scores)))

Training until validation scores don't improve for 250 rounds
Did not meet early stopping. Best iteration is:
[9]	valid_0's ndcg@10: 0.863071
LambdaMART model test score: 0.8489261440498359


### 3.1 : Describe how LambdaMART works. A brief description the model and training objective would be sufficient. (10 points)

**Answer:**\
LambdaMART is a learning-to-rank algorithm that combines LambdaRank with gradient-boosted decision trees. It learns a scoring function for query-document pairs using features such as BM25, VSM, and other engineered signals. Instead of predicting exact relevance labels, it focuses on producing the correct ranking order of documents for each query. The training objective is based on pairwise ranking signals (“lambdas”), which are derived from changes in ranking metrics like NDCG. As a result, the model learns to assign higher scores to more relevant documents, especially ensuring that highly relevant results appear near the top of the ranked list.





### 3.2 : Can we directly optimize **typical** ranking metrics, such as NDCG or MAP? If yes, how? If no, why? (10 points)

***Answer:***/
No, we cannot directly optimize ranking metrics such as NDCG or MAP. This is because these metrics are **non-differientiable** and depend on discrete ranking operations, making them unsuitable for gradient-based optimization.\
\
Instead, learning-to-rank methods use surrogate objectives that approximate these metrics. For example, LambdaMART uses pairwise comparisons and computes "lambda" gradients based on how swapping document pairs would affect NDCG. This allows the model to indirectly optimize ranking quality, even though the actual metric is not directly optimized.

# 4: Think (10 Points)


## 4.1 (5 Points)

How does LambdaMART handle tied relevance scores? What mistakes does ChatGPT (or your choice of LLM) make when answering the question? Maybe the answer is partially correct. What points are missing? Please provide the LLM answer.

***Answer:***\
LambdaMART handles tied relevance scores by treating document pairs with equal relevance as having no preference, meaning no pairwise gradient (“lambda”) is generated for those pairs. As a result, the model does not actively try to order documents with the same relevance label, and their relative order is largely determined by other pairs or noise in the model.

A common mistake LLMs make is to assume that tied documents are either ignored completely or that the model tries to break ties explicitly. While it is true that no direct ranking signal is generated for tied pairs, the model can still assign different scores to them due to interactions with other training pairs. Another missing point is that tied labels reduce the amount of useful training signal, which can limit the model’s ability to learn fine-grained ranking differences.

## 4.2 (5 Points)

If you add more features to a LambdaMART model, will it always improve or maintain performance? What mistakes does ChatGPT (or your choice of LLM) make when answering the question? Maybe the answer is partially correct. What points are missing? Please provide the LLM answer.

***Answer:***\
No, adding more features to a LambdaMART model does not always improve or maintain performance. While additional features can provide more useful signals, they can also introduce noise, redundancy, or irrelevant information, which may degrade model performance or lead to overfitting. The effectiveness of new features depends on their quality and relevance to the ranking task.

A common mistake LLMs make is to assume that more features always improve performance due to the model’s flexibility. However, this ignores issues such as overfitting, feature correlation, and increased model complexity. Another missing point is that some features may be redundant with existing ones (e.g., highly correlated with BM25), providing little additional value. Proper feature selection, regularization, and validation are necessary to ensure that added features actually improve ranking performance.